# M10 · Build BI dashboard views

**Goal:** create (or replace) the three BI dashboard views that the front-end consumes:
1. `marts.v_executive_dashboard` (sql/33) — tenant-month KPIs + MoM deltas + 3-mo rolling
2. `marts.v_operational_dashboard` (sql/34) — daily KPIs + per-100km ratios + 7d rolling
3. `marts.v_maintenance_dashboard` (sql/35) — vehicle-month maintenance leaderboard

Views are stateless — re-running this notebook is a `CREATE OR REPLACE` and never moves data.

In [ ]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

from accent_fleet.db import get_engine, run_sql_file, transaction
from sqlalchemy import text
import pandas as pd

## 2. Inputs

In [ ]:
VIEWS = [
    '33_v_executive_dashboard.sql',
    '34_v_operational_dashboard.sql',
    '35_v_maintenance_dashboard.sql',
]
VIEWS

## 3. Execute

In [ ]:
with transaction() as conn:
    for f in VIEWS:
        run_sql_file(conn, f)
        print(f'ok: {f}')
print('all BI views (re)created.')

## 4. Inspect

In [ ]:
with get_engine().connect() as conn:
    exec_dash = pd.read_sql(text('SELECT * FROM marts.v_executive_dashboard ORDER BY tenant_id, year_month DESC LIMIT 5'), conn)
    op_dash   = pd.read_sql(text('SELECT * FROM marts.v_operational_dashboard ORDER BY fleet_date DESC LIMIT 5'), conn)
    maint_dash = pd.read_sql(text('SELECT * FROM marts.v_maintenance_dashboard ORDER BY year_month DESC, total_cost DESC LIMIT 5'), conn)
display(exec_dash); display(op_dash); display(maint_dash)